# 🧠 Clase 2-B — El Perceptrón: Clasificador lineal paso a paso
### Módulo 2: Deep Learning · Diplomatura en IA · UTN 2026

---
**En este notebook vas a:**
- Construir un perceptrón desde cero con Python puro (sin librerías de ML)
- Ver cómo aprende ajustando sus pesos iterativamente
- Visualizar la línea de decisión moviéndose en tiempo real
- Experimentar con tus propios datos

▶️ **Para ejecutar una celda:** seleccionala y presioná `Shift + Enter`

---

In [ ]:
# Setup — solo necesitamos numpy y matplotlib (ya vienen en Colab)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)
print('✅ Librerías listas')

---
## Parte 1 — El perceptrón manual (sin librerías)

Vamos a implementar exactamente lo que vimos en la presentación:

```
Entradas (x) → Pesos (w) → Suma ponderada (z) → Activación → Salida
```

**Problema:** clasificar si un estudiante aprueba o no según 3 variables.

In [ ]:
# ─── LAS FUNCIONES BÁSICAS ──────────────────────────────────────────

def sigmoid(z):
    """Convierte cualquier número en una probabilidad entre 0 y 1"""
    return 1 / (1 + np.exp(-z))

def predecir(x, w, b):
    """El perceptrón recibe entradas, pondera, suma y activa"""
    z = np.dot(x, w) + b    # suma ponderada
    return sigmoid(z)        # activación

# ─── UN ESTUDIANTE DE EJEMPLO ───────────────────────────────────────
# x1 = horas de estudio (normalizado 0-1)
# x2 = ¿durmió bien? (1=sí, 0=no)
# x3 = ¿comió? (1=sí, 0=no)

x_estudiante = np.array([0.8, 1.0, 1.0])   # estudió mucho, durmió y comió

# Pesos iniciales (todavía no entrenados — aleatorios)
w = np.array([0.1, 0.05, 0.02])
b = -0.1

prob = predecir(x_estudiante, w, b)

print(f'Entradas:         {x_estudiante}')
print(f'Pesos (sin entrenar): {w}')
print(f'Suma ponderada z: {np.dot(x_estudiante, w) + b:.4f}')
print(f'Probabilidad:     {prob:.4f} ({prob*100:.1f}%)')
print(f'Predicción:       {"✅ APRUEBA" if prob >= 0.5 else "❌ NO APRUEBA"}')
print()
print('⚠️  Con pesos no entrenados, la predicción es casi aleatoria.')
print('   Necesitamos ENTRENARLO con ejemplos reales.')

---
## Parte 2 — El loop de entrenamiento

Ahora vamos a entrenar el perceptrón con datos reales.
El loop hace esto en cada iteración:
1. Predice para cada ejemplo
2. Calcula el error
3. Ajusta los pesos con la **regla del perceptrón**: `w_nuevo = w_viejo + η × error × x`


In [ ]:
# ─── DATASET: 8 estudiantes ─────────────────────────────────────────
# [horas_estudio, durmió_bien, comió_bien]
X = np.array([
    [0.9, 1.0, 1.0],   # estudió mucho, durmió, comió
    [0.8, 0.0, 1.0],   # estudió, no durmió, comió
    [0.2, 1.0, 1.0],   # estudió poco, durmió, comió
    [0.1, 0.0, 0.0],   # estudió poco, no durmió, no comió
    [0.7, 1.0, 0.0],   # estudió, durmió, no comió
    [0.3, 0.0, 1.0],   # estudió poco, no durmió, comió
    [0.6, 1.0, 1.0],   # estudió, durmió, comió
    [0.1, 1.0, 0.0],   # estudió poco, durmió, no comió
])

y = np.array([1, 1, 0, 0, 1, 0, 1, 0])  # 1=aprueba, 0=no aprueba

print('Dataset de entrenamiento:')
print('  Horas  Durmió  Comió  |  Aprueba')
print('  -------------------------')
for i in range(len(X)):
    print(f'   {X[i,0]:.1f}    {int(X[i,1])}      {int(X[i,2])}    |    {y[i]}')

In [ ]:
# ─── ENTRENAMIENTO ──────────────────────────────────────────────────

# Inicializar pesos
w = np.zeros(3)
b = 0.0
eta = 0.5    # learning rate (η)
epochs = 100

errores_por_epoca = []
pesos_historia = [w.copy()]

for epoch in range(epochs):
    error_total = 0

    for i in range(len(X)):
        # Predecir
        pred = predecir(X[i], w, b)
        clase_pred = 1 if pred >= 0.5 else 0

        # Error
        error = y[i] - clase_pred
        error_total += abs(error)

        # Ajuste de pesos — la regla del perceptrón
        w = w + eta * error * X[i]
        b = b + eta * error

    errores_por_epoca.append(error_total)
    pesos_historia.append(w.copy())

print('Entrenamiento completo ✅')
print(f'Errores en época 1:   {errores_por_epoca[0]}')
print(f'Errores en época 10:  {errores_por_epoca[9]}')
print(f'Errores en época 100: {errores_por_epoca[-1]}')
print(f'\nPesos finales: {w.round(3)}')
print(f'Sesgo final:   {b:.3f}')

In [ ]:
# ─── CURVA DE APRENDIZAJE ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(errores_por_epoca, color='#06B6D4', linewidth=2.5)
ax.fill_between(range(len(errores_por_epoca)), errores_por_epoca, alpha=0.15, color='#06B6D4')
ax.set_xlabel('Época', fontsize=12)
ax.set_ylabel('Errores totales', fontsize=12)
ax.set_title('Curva de aprendizaje — errores por época', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('✅ La curva bajando = el perceptrón aprendiendo')

In [ ]:
# ─── PREDICCIONES FINALES ───────────────────────────────────────────
print('Predicciones del perceptrón ENTRENADO:')
print('  Horas  Durmió  Comió  |  Real  |  Predicción  |  ¿Correcto?')
print('  -----------------------------------------------------------')
correctos = 0
for i in range(len(X)):
    prob = predecir(X[i], w, b)
    pred = 1 if prob >= 0.5 else 0
    ok = '✅' if pred == y[i] else '❌'
    if pred == y[i]: correctos += 1
    print(f'   {X[i,0]:.1f}    {int(X[i,1])}      {int(X[i,2])}    |   {y[i]}    |   {pred} ({prob:.2f})    |   {ok}')
print(f'\nPrecisión: {correctos}/{len(X)} = {correctos/len(X)*100:.0f}%')

---
## Parte 3 — El gráfico de la línea de decisión

Para poder visualizar en 2D, vamos a usar solo 2 características: **horas de estudio** y **si durmió bien**.

La línea de decisión es la frontera donde el perceptrón cambia de "aprueba" a "no aprueba".

In [ ]:
# Dataset simplificado 2D para visualización
X2 = np.array([
    [0.9, 1.0], [0.8, 0.0], [0.2, 1.0], [0.1, 0.0],
    [0.7, 1.0], [0.3, 0.0], [0.6, 1.0], [0.1, 1.0],
    [0.75, 0.5], [0.4, 0.8], [0.5, 0.2], [0.85, 0.9]
])
y2 = np.array([1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1])

# Entrenar perceptrón 2D para visualizar la frontera
w2 = np.zeros(2)
b2 = 0.0
snapshots = []   # guardar pesos en distintas épocas

for epoch in range(150):
    for i in range(len(X2)):
        pred = 1 if predecir(X2[i], w2, b2) >= 0.5 else 0
        error = y2[i] - pred
        w2 = w2 + 0.3 * error * X2[i]
        b2 = b2 + 0.3 * error
    if epoch in [0, 5, 20, 149]:
        snapshots.append((epoch+1, w2.copy(), b2))

# Graficar 4 momentos del aprendizaje
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('La línea de decisión moviéndose durante el entrenamiento', fontsize=13, fontweight='bold')

colors = ['#F87171' if label == 0 else '#4ADE80' for label in y2]

for idx, (epoch, ws, bs) in enumerate(snapshots):
    ax = axes[idx]
    ax.scatter(X2[:, 0], X2[:, 1], c=colors, s=100, edgecolors='black', zorder=5)

    # Línea de decisión
    x_line = np.linspace(-0.1, 1.1, 100)
    if abs(ws[1]) > 1e-6:
        y_line = -(ws[0] * x_line + bs) / ws[1]
        ax.plot(x_line, y_line, 'b-', linewidth=2.5, alpha=0.8)
    else:
        ax.axvline(x=-bs/ws[0] if abs(ws[0])>1e-6 else 0.5, color='blue', linewidth=2.5)

    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(-0.2, 1.3)
    ax.set_xlabel('Horas de estudio', fontsize=9)
    ax.set_ylabel('¿Durmió bien?', fontsize=9)
    ax.set_title(f'Época {epoch}', fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3)

    red_patch = mpatches.Patch(color='#F87171', label='No aprueba')
    green_patch = mpatches.Patch(color='#4ADE80', label='Aprueba')
    ax.legend(handles=[green_patch, red_patch], fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('decision_boundary.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ La línea azul es la frontera de decisión del perceptrón.')

---
## Parte 4 — Tu propio dataset ✏️

Ahora es tu turno. **Modificá los datos** y observá cómo cambia la línea de decisión.

Podés cambiar:
- Los valores de `X_propio` (entre 0.0 y 1.0)
- Las etiquetas `y_propio` (1 = aprueba, 0 = no aprueba)
- El `learning_rate` (probá con 0.01, 0.1, 0.5, 1.0)

**Preguntas para explorar:**
- ¿Qué pasa si todos los que estudiaron 0.8+ aprueban y los demás no?
- ¿Y si el perceptrón tiene datos contradictorios (mismo input, distinto output)?
- ¿Qué learning rate converge más rápido?

In [ ]:
# ✏️ MODIFICÁ ESTOS VALORES Y RE-EJECUTÁ LA CELDA

X_propio = np.array([
    [0.9, 1.0],   # ← [horas_estudio, durmió_bien]
    [0.1, 1.0],
    [0.8, 0.0],
    [0.2, 0.0],
    [0.6, 1.0],
    [0.3, 0.5],
])

y_propio = np.array([1, 0, 1, 0, 1, 0])  # ← 1=aprueba, 0=no aprueba

learning_rate = 0.3  # ← probá con 0.01, 0.1, 0.5, 1.0

# ─── No modifiques desde acá ────────────────────────────────────────
w_p = np.zeros(2)
b_p = 0.0
errores_p = []

for epoch in range(200):
    err = 0
    for i in range(len(X_propio)):
        pred = 1 if predecir(X_propio[i], w_p, b_p) >= 0.5 else 0
        error = y_propio[i] - pred
        err += abs(error)
        w_p = w_p + learning_rate * error * X_propio[i]
        b_p = b_p + learning_rate * error
    errores_p.append(err)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Curva de aprendizaje
ax1.plot(errores_p, color='#06B6D4', linewidth=2)
ax1.set_title('Curva de aprendizaje', fontweight='bold')
ax1.set_xlabel('Época'); ax1.set_ylabel('Errores'); ax1.grid(alpha=0.3)

# Frontera de decisión final
colors_p = ['#F87171' if l == 0 else '#4ADE80' for l in y_propio]
ax2.scatter(X_propio[:, 0], X_propio[:, 1], c=colors_p, s=150, edgecolors='black', zorder=5)
x_l = np.linspace(-0.1, 1.1, 100)
if abs(w_p[1]) > 1e-6:
    y_l = -(w_p[0] * x_l + b_p) / w_p[1]
    ax2.plot(x_l, y_l, 'b-', linewidth=2.5)
ax2.set_xlim(-0.1, 1.1); ax2.set_ylim(-0.2, 1.3)
ax2.set_xlabel('x₁'); ax2.set_ylabel('x₂')
ax2.set_title('Frontera de decisión final', fontweight='bold')
ax2.grid(alpha=0.3)

plt.suptitle(f'Tu perceptrón (η={learning_rate})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Accuracy
correctos_p = sum(1 for i in range(len(X_propio)) if (1 if predecir(X_propio[i], w_p, b_p) >= 0.5 else 0) == y_propio[i])
print(f'\nPrecisión final: {correctos_p}/{len(X_propio)} = {correctos_p/len(X_propio)*100:.0f}%')
print(f'Pesos aprendidos: {w_p.round(3)} | Sesgo: {b_p:.3f}')

---
## Resumen de lo aprendido hoy

| Concepto | En una oración |
|----------|----------------|
| **Perceptrón** | Neurona artificial que calcula `z = w·x + b` y aplica una activación |
| **Entrenamiento** | Ajustar los pesos usando la regla: `w = w + η × error × x` |
| **Learning rate (η)** | Qué tan grande es cada paso de ajuste (ni muy grande ni muy pequeño) |
| **Frontera de decisión** | La línea que separa las dos clases en el espacio de los datos |
| **Limitación** | Solo funciona con problemas linealmente separables (como AND, no XOR) |
| **Solución** | Apilar perceptrones en capas → Red Neuronal Multicapa |

---
**Próxima clase:** Funciones de pérdida y métricas de evaluación